# The stochastic maze as an experimental laboratory

The maze is useful because its latent stochastic process can be made exactly
finite without becoming trivial. We will isolate action noise, spatial
heterogeneity, several wall processes, hazards, partial observation, and
temporal nonstationarity. Whenever the assumptions permit, we construct the
exact kernel and solve the corresponding MDP.

The notebook uses the convention north/east/south/west = 0/1/2/3. A wall is
an *edge* between adjacent cells; a blocked cell is removed from navigation.


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rllab.environments import (
    EventWall,
    ExactModelUnavailable,
    Hazard,
    IndependentWall,
    MarkovWall,
    MazeAction,
    MovingHazard,
    NonstationarityConfig,
    ScheduledWall,
    StochasticMazeEnv,
)
from rllab.theory import value_iteration
from rllab.visualization import (
    animate_topology,
    plot_maze,
    plot_policy,
    plot_transition_noise,
)

SEED = 17
available_styles = set(plt.style.available)
plot_style = next(
    (
        style
        for style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot")
        if style in available_styles
    ),
    "default",
)
plt.style.use(plot_style)


## 1. Deterministic baseline

With reliability one, no dynamic walls, deterministic rewards, and full state
observation, the environment is a conventional finite MDP. `reset(seed=...)`
seeds every stochastic component; `info` still reports the latent wall,
hazard, and regime state so an experiment recorder can retain it.


In [ ]:
deterministic = StochasticMazeEnv(
    shape=(5, 7),
    start=(4, 0),
    goals={(0, 6): 8.0},
    blocked_cells={(1, 1), (1, 2), (3, 4), (3, 5)},
    static_walls={((2, 2), (2, 3)), ((1, 5), (2, 5))},
    action_reliability=1.0,
    step_reward=-0.04,
    max_episode_steps=150,
)
observation, reset_info = deterministic.reset(seed=SEED)
print("initial observation:", observation)
print("initial coordinate:", deterministic.index_to_state[int(observation)])
print("diagnostic fields:", sorted(reset_info))
plot_maze(deterministic, title="Deterministic topology")
plt.show()


## 2. Stochastic action channel

If action $a$ is intended, the realized action is sampled from a configurable
channel. For the common relative-slip parameterization,

$$P(A^{real}=a\mid s,a)=p(s,a),$$

and the remaining mass is assigned to left, right, stay, or backward motion.
Collision with a boundary, cell, or active edge wall maps any attempted move
back to the current state. The exact kernel therefore combines action-channel
randomness with topology-induced aggregation.


In [ ]:
slippery = StochasticMazeEnv(
    shape=(4, 5),
    start=(3, 0),
    goals={(0, 4): 5.0},
    action_reliability=0.70,
    slip_weights={"left": 0.35, "right": 0.35, "stay": 0.20, "backward": 0.10},
    step_reward=-0.03,
    max_episode_steps=100,
)
P, R = slippery.transition_reward_kernels()
start = slippery.state_to_index[(3, 0)]
north = int(MazeAction.NORTH)
support = np.flatnonzero(P[start, north] > 0)
labels = [str(slippery.index_to_state[int(state)]) for state in support]
probabilities = P[start, north, support]

fig, ax = plt.subplots(figsize=(7, 3.3))
ax.bar(labels, probabilities)
ax.set(
    title="Exact next-state distribution for intended north",
    xlabel="next coordinate",
    ylabel="probability",
    ylim=(0, 1),
)
for x, probability in enumerate(probabilities):
    ax.text(x, probability + 0.025, f"{probability:.2f}", ha="center")
plt.show()
assert np.allclose(P.sum(axis=2), 1.0)


## 3. Spatially heterogeneous transition noise

Reliability may be scalar, state-dependent, or state--action-dependent. A
scalar is the baseline; sparse maps override it. This lets a region represent
ice, wind, intermittent control, or an uncertain actuator without changing
the agent interface.


In [ ]:
reliability_map = {
    (row, column): 0.52 + 0.06 * row
    for row in range(1, 4)
    for column in range(2, 5)
}
heterogeneous = StochasticMazeEnv(
    shape=(5, 7),
    start=(4, 0),
    goals={(0, 6): 8.0},
    action_reliability=0.96,
    state_reliability=reliability_map,
    slip_weights={"left": 0.45, "right": 0.45, "stay": 0.10},
    step_reward=-0.04,
)
plot_transition_noise(heterogeneous, title="Reliability $p(s)$")
plt.show()


## 4. Walls are stochastic processes

Four dynamic mechanisms answer different questions:

- independent walls resample a Bernoulli presence state;
- Markov walls retain temporal correlation through $p_{01}$ and $p_{11}$;
- scheduled walls impose known interventions;
- event walls change after the agent reaches a trigger set.

Static edges may coexist with all four. The environment updates walls at a
documented point in its step transition and reports structural events.


In [ ]:
edge_independent = ((1, 1), (1, 2))
edge_markov = ((2, 2), (2, 3))
edge_scheduled = ((3, 1), (3, 2))
edge_event = ((3, 3), (3, 4))
dynamic = StochasticMazeEnv(
    shape=(5, 6),
    start=(4, 0),
    goals={(0, 5): 5.0},
    action_reliability=0.85,
    independent_walls=[IndependentWall(edge_independent, presence_probability=0.25)],
    markov_walls=[MarkovWall(edge_markov, p01=0.12, p11=0.88, initial_probability=0.5)],
    scheduled_walls=[ScheduledWall(edge_scheduled, changes={10: True, 25: False})],
    event_walls=[
        EventWall(
            edge_event,
            trigger_states={(4, 0)},
            present_after_trigger=True,
            initial_present=False,
            once=True,
        )
    ],
    max_episode_steps=80,
)

observation, _ = dynamic.reset(seed=SEED)
wall_history = [tuple(dynamic.current_walls)]
scalar_info = []
for _ in range(35):
    action = dynamic.action_space.sample()
    observation, reward, terminated, truncated, info = dynamic.step(action)
    wall_history.append(tuple(dynamic.current_walls))
    scalar_info.append({key: value for key, value in info.items() if np.isscalar(value)})
    if terminated or truncated:
        observation, _ = dynamic.reset()

fig, axes = plt.subplots(1, 4, figsize=(16, 3.7))
for time, ax in zip((0, 10, 20, 30), axes, strict=True):
    plot_maze(dynamic, walls=wall_history[time], ax=ax, title=f"walls at t={time}")
plt.tight_layout()
plt.show()

# In Jupyter, display this object to animate every recorded realization.
topology_animation = animate_topology(dynamic, wall_history, interval=180)[2]


### Markovization by state augmentation

A maze position alone is not Markov when a persistent wall state is latent
from the state. For $m$ binary Markov walls, the exact state is
$(s,w_1,\ldots,w_m)$ and has up to $|\mathcal S|2^m$ elements. The environment
builds this augmented MDP only when requested and tractable; it never calls a
marginal one-step kernel an exact solution to the persistent process.


In [ ]:
one_markov_wall = StochasticMazeEnv(
    shape=(3, 4),
    start=(2, 0),
    goals={(0, 3): 4.0},
    action_reliability=0.9,
    markov_walls=[
        MarkovWall(((1, 1), (1, 2)), p01=0.08, p11=0.94, initial_probability=0.4)
    ],
    step_reward=-0.02,
)
augmented_mdp = one_markov_wall.exact_mdp(augment_walls=True)
augmented_solution = value_iteration(augmented_mdp, gamma=0.98)
print(
    f"physical cells={np.prod(one_markov_wall.shape)}, "
    f"augmented states={augmented_mdp.n_states}, "
    f"Bellman sweeps={augmented_solution.iterations}"
)


## 5. Hazards and partial observation

Hazards can be terminal, penalizing, probabilistically active, or moving.
Observation modes are separate from latent dynamics: `state` returns an exact
integer index, `full` exposes a structured diagnostic observation, `local`
returns a finite neighborhood with noisy wall readings, and `noisy_state`
corrupts the reported position. Tabular exact-control comparisons in this
repository use the fully observed state mode unless the latent state is
explicitly augmented.


In [ ]:
hazard_kwargs = dict(
    shape=(5, 7),
    start=(4, 0),
    goals={(0, 6): 7.0},
    hazards=[
        Hazard(
            (1, 4),
            penalty=-5.0,
            terminal=True,
            activation_probability=0.8,
            terminal_probability=1.0,
        )
    ],
    moving_hazards=[
        MovingHazard((3, 3), penalty=-2.0, terminal=False, movement_probability=0.7)
    ],
    action_reliability=0.9,
    max_episode_steps=100,
)
local_env = StochasticMazeEnv(
    **hazard_kwargs,
    observation_mode="local",
    observation_radius=1,
    wall_observation_noise=0.10,
)
noisy_env = StochasticMazeEnv(
    **hazard_kwargs,
    observation_mode="noisy_state",
    state_observation_noise=0.15,
)
local_observation, local_info = local_env.reset(seed=SEED)
noisy_observation, noisy_info = noisy_env.reset(seed=SEED)
print("local observation space:", local_env.observation_space)
print("local observation:", local_observation)
print("noisy-state observation / latent state:", noisy_observation, noisy_info["state_index"])
plot_maze(local_env, title="Hazards remain part of the latent process")
plt.show()


## 6. Nonstationarity

The transition and reward kernels may drift gradually, jump at an abrupt
change point, switch periodically, or switch at geometrically distributed
times. The regime is recorded in `info`. Such a run has no single stationary
$Q^*$; evaluation must use a time-indexed oracle, instantaneous frozen model,
or a tracking metric, and label that choice.


In [ ]:
regimes = {
    "gradual": NonstationarityConfig(
        mode="gradual",
        reliability_multipliers=(1.0, 0.65),
        reward_multipliers=(1.0, 1.0),
        horizon=120,
    ),
    "abrupt": NonstationarityConfig(
        mode="abrupt",
        reliability_multipliers=(1.0, 0.65),
        reward_multipliers=(1.0, 1.0),
        change_step=50,
    ),
    "periodic": NonstationarityConfig(
        mode="periodic",
        reliability_multipliers=(1.0, 0.65),
        reward_multipliers=(1.0, 1.0),
        period=25,
    ),
    "random": NonstationarityConfig(
        mode="random",
        reliability_multipliers=(1.0, 0.65),
        reward_multipliers=(1.0, 1.0),
        switch_probability=0.02,
    ),
}
regime_rows = []
for name, regime in regimes.items():
    env = StochasticMazeEnv(
        shape=(4, 6),
        start=(3, 0),
        goals={(0, 5): 5.0},
        action_reliability=0.92,
        nonstationarity=regime,
        max_episode_steps=140,
    )
    env.reset(seed=SEED)
    for time in range(120):
        _, _, terminated, truncated, info = env.step(env.action_space.sample())
        scalars = {key: value for key, value in info.items() if np.isscalar(value)}
        regime_rows.append({"mechanism": name, "time": time, **scalars})
        if terminated or truncated:
            env.reset()
regime_frame = pd.DataFrame(regime_rows)
candidate = next(
    (column for column in regime_frame if "reliability" in column and "multiplier" in column),
    None,
)
if candidate is not None:
    for name, sample in regime_frame.groupby("mechanism"):
        plt.plot(sample["time"], sample[candidate], label=name)
    plt.ylabel(candidate.replace("_", " "))
    plt.xlabel("environment step")
    plt.legend()
    plt.title("Recorded transition regime")
    plt.show()
else:
    display(regime_frame.head())


## 7. Exact policies under increasing action noise

Consider two routes. The center corridor is short, but lateral slips enter
terminal hazards. The lower route is longer and shielded by edge walls. At
high reliability, the direct route dominates; as control deteriorates, the
value of physical separation from the hazards can exceed the step cost.


In [ ]:
corridor_hazards = [
    Hazard((row, column), penalty=-8.0, terminal=True)
    for row in (1, 3)
    for column in range(2, 7)
]
protected_edges = {
    ((3, column), (4, column))
    for column in range(1, 8)
}


def risk_safe_maze(reliability):
    return StochasticMazeEnv(
        shape=(5, 9),
        start=(2, 0),
        goals={(2, 8): 8.0},
        hazards=corridor_hazards,
        static_walls=protected_edges,
        action_reliability=reliability,
        slip_weights={"left": 0.5, "right": 0.5},
        step_reward=-0.06,
        max_episode_steps=250,
    )


reliabilities = (1.0, 0.90, 0.75, 0.60)
solved = []
fig, axes = plt.subplots(1, len(reliabilities), figsize=(18, 4.0))
for reliability, ax in zip(reliabilities, axes, strict=True):
    env = risk_safe_maze(reliability)
    solution = value_iteration(env.exact_mdp(), gamma=0.98)
    solved.append((env, solution))
    plot_policy(
        solution.policy,
        env,
        values=solution.values,
        ax=ax,
        title=f"p={reliability:.2f}",
    )
plt.tight_layout()
plt.show()


In [ ]:
reliability_grid = np.linspace(0.55, 1.0, 24)
direct_advantage = []
selected_action = []
for reliability in reliability_grid:
    env = risk_safe_maze(float(reliability))
    solution = value_iteration(env.exact_mdp(), gamma=0.98)
    start = env.state_to_index[(2, 0)]
    direct_advantage.append(
        solution.q_values[start, int(MazeAction.EAST)]
        - solution.q_values[start, int(MazeAction.SOUTH)]
    )
    selected_action.append(solution.policy[start])

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.axhline(0, color="0.2", linewidth=1)
ax.plot(reliability_grid, direct_advantage, marker="o", markersize=3)
ax.set(
    xlabel="intended-action reliability",
    ylabel=r"$Q^*(s_0, east)-Q^*(s_0, south)$",
    title="Exact preference: short risky route versus longer protected route",
)
plt.show()


## 8. Exactness boundaries

`exact_mdp()` is valid for a stationary, fully observed configuration. Markov
wall memory can be made exact with explicit augmentation, at exponential cost.
Scheduled/event walls, moving hazards, parameter drift, episode-level random
parameters, and corrupted observations generally require additional time,
event, hazard, parameter, or belief state. The environment raises
`ExactModelUnavailable` when the requested finite model would silently omit
such state. This distinction is essential: simulation access is not exact
model access.
